# 9. Memory System

MARES has a two-tier memory system:
1. **AgentMemory** — per-agent conversation history (bounded deque)
2. **SharedMemory** — cross-agent key/value store with output indexing
3. **MemoryStore** — optional persistent backend (in-memory or Redis)


In [ ]:
from mares.memory.agent_memory import AgentMemory

# Per-agent memory with bounded history
memory = AgentMemory(agent_name="planner_agent", capacity=100)

memory.add("user", "Analyze Celery failures")
memory.add("assistant", '{"tasks": [...]}')
memory.add("user", "Focus on Redis broker issues")
memory.add("assistant", "OK, diving deeper into Redis")

print(f"History ({len(memory)} messages):")
for msg in memory.history():
    print(f"  [{msg['role']}] {msg['content'][:60]}...")

## Shared Memory

`SharedMemory` allows agents to share data during a run:

In [ ]:
from mares.memory.shared_memory import SharedMemory
from mares.models.agent_output import AgentOutput

shared = SharedMemory()

# Store outputs by sub-task ID
await shared.store_output(
    AgentOutput(agent="research_agent", sub_task_id=1, content="finding-1")
)
await shared.store_output(
    AgentOutput(agent="execution_agent", sub_task_id=2, content="code-output")
)

# Retrieve individual outputs
out1 = await shared.get_output(1)
out2 = await shared.get_output(2)
print(f"Output 1: agent={out1.agent}, content={out1.content}")
print(f"Output 2: agent={out2.agent}, content={out2.content}")

# Key/value storage
await shared.set("task_description", "Analyze Celery")
val = await shared.get("task_description")
print(f"KV store: {val}")

## Persistent Storage

`MemoryStore` provides an optional Redis backend with automatic fallback to in-memory:

In [ ]:
from mares.memory.memory_store import MemoryStore

# In-memory store (default)
store = MemoryStore(backend="memory")
await store.set("key1", {"nested": "value", "count": 42})
value = await store.get("key1")
print(f"Stored value: {value}")

# Redis store (if REDIS_URL is set)
import os
redis_url = os.getenv("REDIS_URL")
if redis_url:
    redis_store = MemoryStore(backend="redis", redis_url=redis_url)
    await redis_store.set("test_key", "working")
    val = await redis_store.get("test_key")
    print(f"Redis store works: {val}")
    await redis_store.delete("test_key")
else:
    print("No REDIS_URL set — skipping Redis test")